In [ ]:
import pandas as pd
import os
from helper_functions import strings2lists

results_path = r"D:\DATA\abmil_checkpoints\inference_hopt_binary.csv"
metadata_path =  r"D:\DATA\EXP3_abmil\abmil_inference_M_idx.csv"

results_df = pd.read_csv(results_path)
print("Results columns:", results_df.columns)
meta_df = pd.read_csv(metadata_path)
print("Metadata columns:", meta_df.columns)


list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    meta_df[col] = meta_df[col].apply(strings2lists)

# Join results with metadata
df = results_df.merge(
    meta_df,
    left_on="slide_path",
    right_on="filename",
    how="inner",
    suffixes=("_results", "_meta"),
)

print("Tissue distribution:\n", df["T_category"].value_counts())

In [ ]:
# Name of the true label column in your metadata
true_label_col = "M_idx"

# Keep only slides where prediction matches true label
df_correct = df[df["pred_label"] == df[true_label_col]].copy()

# Keep only high-confidence predictions, e.g. >= 0.95
df_correct = df_correct[df_correct["confidence"] >= 0.95].copy()

print("Correct + high-conf shape:", df_correct.shape)
print("Tissue distribution (correct, high-conf):\n", df_correct["T_category"].value_counts())


In [ ]:
# Convert single-element lists to scalars
def extract_tissue(x):
    if isinstance(x, (list, tuple)) and len(x) == 1:
        return x[0]
    return x

df_correct = df_correct.copy()
df_correct["T_category"] = df_correct["T_category"].apply(extract_tissue)

# Now compute top 5 tissues
top5_tissues = df_correct["T_category"].value_counts().head(5).index.tolist()
print("Top 5 tissues (correct + high-conf):\n", top5_tissues)

# Pick one slide per tissue
audit_slides = []

for t in top5_tissues:
    sub = df_correct[df_correct["T_category"] == t]
    slide_row = sub.sample(n=1, random_state=42).iloc[0]
    audit_slides.append(slide_row)

audit_df = pd.DataFrame(audit_slides)
print("Audit slides:\n", audit_df[["slide_path", "T_category", true_label_col, "pred_label", "confidence"]])

In [ ]:
audit_df.to_csv(r"D:\DATA\audit\phase1_audit_slides.csv", index=False)

In [ ]:
from tile_audit import TileAuditor

checkpoint_path = r"D:\DATA\abmil_checkpoints\abmil_hopt_binary.pt"
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

auditor = TileAuditor(
    checkpoint_path=checkpoint_path,
    zarr_dir=zarr_dir,
    feature_key="features_h-optimus-0",
    tile_key="tiles_224",
    top_k=10,
)

for _, row in audit_df.iterrows():
    slide_path = row["slide_path"]
    tissue = row["T_category"]
    print("\n=== Slide:", slide_path, "Tissue:", tissue)
    auditor.audit_slide(slide_path, tissue=tissue)


In [ ]:
# Load one known slide’s features and run inference

import os
import numpy as np
import torch
from wsidata import open_wsi
import pandas as pd
from abmil import load_checkpoint

slides = df_inference["filename"].tolist()
slide_path = slides[0]

zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"
checkpoint_path = r"D:\DATA\abmil_checkpoints\abmil_hopt_binary.pt"

# These should match the configuration used during training.
feature_key = "features_h-optimus-0"
tile_key = "tiles_224"

# Build the feature-store path exactly as your pipeline does.
zarr_path = os.path.join(zarr_dir, os.path.basename(slide_path).replace(".mrxs", ".zarr"))

# Open the WSI with its Zarr backing store and load tile features.
wsi = open_wsi(slide_path, zarr_path)
adata = wsi.tables[feature_key]

features = torch.from_numpy(adata.X).float()       # [N_tiles, 1536]
tile_ids = np.asarray(adata.obs["tile_id"])

print("Features shape:", tuple(features.shape))
print("Number of tile IDs:", len(tile_ids))
print("First tile IDs:", tile_ids[:5])

# Load your existing checkpoint using the function in your pipeline.
model, config, label_mapping = load_checkpoint(checkpoint_path)

device = next(model.parameters()).device
model.eval()

# Run inference. Use float32 here for exact reconstruction later;
# do not use autocast for this audit cell.
with torch.inference_mode():
    logits, attention = model(features.to(device))

probabilities = torch.softmax(logits.float(), dim=0).cpu().numpy()
predicted_class = int(np.argmax(probabilities))

print("Logits:", logits.float().cpu().numpy())
print("Probabilities:", probabilities)
print("Predicted class:", predicted_class)
print("Attention shape:", tuple(attention.shape))
print("Attention column sums:", attention.float().sum(dim=0).cpu().numpy())

In [ ]:
# Compute additive tile contributions and verify reconstruction

# Get the device the model is on (from its parameters)
device = next(model.parameters()).device

# Ensure we are in float32 for an exact check.
with torch.inference_mode():
    # Use float32 features even if the model internally uses float16.
    feats = features.to(device).float()      # [N, D]
    attn  = attention.to(device).float()     # [N, n_heads]

    # Average across heads to match how the model pools them.
    a = attn.mean(dim=1)                     # [N]

    W = model.classifier.weight.float()      # [C, D]
    b = model.classifier.bias.float()        # [C]

    # Per-tile, per-class contributions: C_{k,c} = a_k * (W_c · h_k)
    contrib = a[:, None] * (feats @ W.T)     # [N, C]

    # Reconstruct logits as sum of contributions plus bias.
    logits_recon = contrib.sum(dim=0) + b    # [C]

    logits_orig = model(feats)[0].float()    # [C]

    max_error = (logits_recon - logits_orig).abs().max().item()

print("Original logits:   ", logits_orig.cpu().numpy())
print("Reconstructed:     ", logits_recon.cpu().numpy())
print("Max absolute error:", max_error)
print("Contribution shape:", tuple(contrib.shape))

# Treat contrib[:, 0]  as “evidence for class 0” and  contrib[:, 1]  as “evidence for class 1” for this slide.

In [ ]:
# Compute additive tile contributions and define tile sets

device = next(model.parameters()).device

with torch.inference_mode():
    feats = features.to(device).float()          # [N, D]
    attn  = attention.to(device).float()         # [N, n_heads]

    a = attn.mean(dim=1)                         # [N]

    W = model.classifier.weight.float()          # [C, D]
    b = model.classifier.bias.float()            # [C]

    # C_{k,c} = a_k * (W_c · h_k)
    # W is [C, D], so W.T is [D, C]
    contrib_torch = a[:, None] * (feats @ W.T)   # [N, C]

    # Convert to NumPy once, after all torch ops are done.
    contrib = contrib_torch.cpu().numpy()        # [N, C]
    a_np = a.cpu().numpy()                       # [N]

# For this slide, class 1 is the predicted / "disease" class.
c_disease = 1

# 1) Top disease-evidence tiles
disease_score = contrib[:, c_disease]
top_k = 20
top_disease_idx = np.argsort(disease_score)[::-1][:top_k]

# 2) Neutral tiles: near-zero evidence for both classes
neutrality_score = -np.abs(contrib).max(axis=1)
top_neutral_idx = np.argsort(neutrality_score)[::-1][:top_k]

# 3) Low-attention tiles
low_attn_idx = np.argsort(a_np)[:top_k]

# Build a small DataFrame for inspection.
tile_df = pd.DataFrame({
    "tile_id": tile_ids,
    "attention": a_np,
    "contrib_class0": contrib[:, 0],
    "contrib_class1": contrib[:, 1],
    "disease_score": disease_score,
    "neutrality_score": neutrality_score,
})

tile_df["set"] = "other"
tile_df.loc[top_disease_idx, "set"] = "disease_top"
tile_df.loc[top_neutral_idx, "set"] = "neutral"
tile_df.loc[low_attn_idx, "set"] = "low_attention"

print(tile_df["set"].value_counts())
print("\nTop 5 disease tiles:")
print(tile_df.loc[top_disease_idx, ["tile_id", "disease_score", "attention"]].head())

print("\nTop 5 neutral tiles:")
print(tile_df.loc[top_neutral_idx, ["tile_id", "neutrality_score", "attention"]].head())

print("\nTop 5 low-attention tiles:")
print(tile_df.loc[low_attn_idx, ["tile_id", "attention", "disease_score"]].head())

# Result: Disease tiles: small but clearly positive  disease_score , with modest attention.
# Neutral tiles:  neutrality_score  essentially zero (≈ 1e-8), tiny attention.
# Low-attention tiles: even smaller attention, also near-zero disease score.
# That confirms the contribution logic is working and that “neutral” and “low attention” are not identical sets.